In [57]:
import numpy as np
# ----- JAX IN ------------------
import haiku as hk
import chex
from typing import Any, Callable, Mapping, Optional, List
from InteractionNet import deep_typed_graph_net
from InteractionNet import typed_graph
from InteractionNet import grid_mesh_connectivity  # Assuming you have the connectivity utility
from InteractionNet import model_utils  # Assuming you have the utility for feature computation
from InteractionNet import icosahedral_mesh

## IN JAX
---

In [12]:
@chex.dataclass(frozen=True, eq=True)
class ModelConfig:
  """Defines the architecture of the GraphCast neural network architecture.

  Properties:
    resolution: The resolution of the data, in degrees (e.g. 0.25 or 1.0).
    mesh_size: How many refinements to do on the multi-mesh.
    gnn_msg_steps: How many Graph Network message passing steps to do.
    latent_size: How many latent features to include in the various MLPs.
    hidden_layers: How many hidden layers for each MLP.
    radius_query_fraction_edge_length: Scalar that will be multiplied by the
        length of the longest edge of the finest mesh to define the radius of
        connectivity to use in the Grid2Mesh graph. Reasonable values are
        between 0.6 and 1. 0.6 reduces the number of grid points feeding into
        multiple mesh nodes and therefore reduces edge count and memory use, but
        1 gives better predictions.
    mesh2grid_edge_normalization_factor: Allows explicitly controlling edge
        normalization for mesh2grid edges. If None, defaults to max edge length.
        This supports using pre-trained model weights with a different graph
        structure to what it was trained on.
  """
  resolution: float
  mesh_size: int
  latent_size: int
  gnn_msg_steps: int
  hidden_layers: int
  radius_query_fraction_edge_length: float
  mesh2grid_edge_normalization_factor: Optional[float] = None

In [39]:
# Instantiate the model config with the desired parameters.
model_config = ModelConfig(
    resolution=0.25,
    mesh_size=3,
    latent_size=128,
    gnn_msg_steps=3,
    hidden_layers=2,
    radius_query_fraction_edge_length=0.6,
    mesh2grid_edge_normalization_factor=None
)

In [58]:
def GraphCastIN(input_graph: typed_graph.TypedGraph):
    # Create the model object and call it to perform a forward pass
    GNN = deep_typed_graph_net.DeepTypedGraphNet(
        embed_nodes=True,  # Embed raw features of the grid and mesh nodes.
        embed_edges=True,  # Embed raw features of the grid2mesh edges.
        edge_latent_size=dict(grid2mesh=model_config.latent_size),
        node_latent_size=dict(
            mesh_nodes=model_config.latent_size,
            grid_nodes=model_config.latent_size),
        mlp_hidden_size=model_config.latent_size,
        mlp_num_hidden_layers=model_config.hidden_layers,
        num_message_passing_steps=1,
        use_layer_norm=True,
        include_sent_messages_in_node_update=False,
        activation="swish",
        f32_aggregation=True,
        aggregate_normalization=None,
        name="grid2mesh_gnn",
    )
    return GNN(input_graph)

# Initialize the Haiku module with hk.transform
transformed_model = hk.transform(GraphCastIN)

# Initialize a random number generator (rng) for parameter initialization
rng = jax.random.PRNGKey(42)

# Example input graph; this should be a valid `typed_graph.TypedGraph` instance
input_graph = ...  # Your graph data (TypedGraph) goes here

# Initialize parameters using Haiku
params = transformed_model.init(rng, input_graph)

# Now, run a forward pass using the initialized parameters
output = transformed_model.apply(params, rng, input_graph)

AttributeError: 'ellipsis' object has no attribute 'edges'

In [61]:
def init_grid2mesh_graph(
    grid_lat: np.ndarray,
    grid_lon: np.ndarray,
    finest_mesh: List,
    query_radius: float,
    grid_nodes_lat: np.ndarray,
    grid_nodes_lon: np.ndarray,
    mesh_nodes_lat: np.ndarray,
    mesh_nodes_lon: np.ndarray,
    num_grid_nodes: int,
    num_mesh_nodes: int,
    spatial_features_kwargs: dict,
) -> typed_graph.TypedGraph:
    """Build Grid2Mesh graph."""
    
    # Create some edges according to distance between mesh and grid nodes.
    assert grid_lat is not None and grid_lon is not None
    grid_indices, mesh_indices = grid_mesh_connectivity.radius_query_indices(
        grid_latitude=grid_lat,
        grid_longitude=grid_lon,
        mesh=_meshes[-1],
        radius=query_radius,
    )

    # Edges sending info from grid to mesh.
    senders = grid_indices
    receivers = mesh_indices

    # Precompute structural node and edge features according to config options.
    (senders_node_features, receivers_node_features, edge_features) = model_utils.get_bipartite_graph_spatial_features(
        senders_node_lat=grid_nodes_lat,
        senders_node_lon=grid_nodes_lon,
        receivers_node_lat=mesh_nodes_lat,
        receivers_node_lon=mesh_nodes_lon,
        senders=senders,
        receivers=receivers,
        edge_normalization_factor=None,
        **spatial_features_kwargs,
    )

    n_grid_node = np.array([num_grid_nodes])
    n_mesh_node = np.array([num_mesh_nodes])
    n_edge = np.array([mesh_indices.shape[0]])

    # Create NodeSet for grid and mesh nodes
    grid_node_set = typed_graph.NodeSet(
        n_node=n_grid_node, features=senders_node_features
    )
    mesh_node_set = typed_graph.NodeSet(
        n_node=n_mesh_node, features=receivers_node_features
    )

    # Create EdgeSet
    edge_set = typed_graph.EdgeSet(
        n_edge=n_edge,
        indices=typed_graph.EdgesIndices(senders=senders, receivers=receivers),
        features=edge_features,
    )

    # Create Node and Edge Dictionaries
    nodes = {"grid_nodes": grid_node_set, "mesh_nodes": mesh_node_set}
    edges = {
        typed_graph.EdgeSetKey("grid2mesh", ("grid_nodes", "mesh_nodes")): edge_set
    }

    # Create the TypedGraph object
    grid2mesh_graph = typed_graph.TypedGraph(
        context=typed_graph.Context(n_graph=np.array([1]), features=()),
        nodes=nodes,
        edges=edges,
    )

    return grid2mesh_graph

In [ ]:
# Init mesh properties
_meshes = icosahedral_mesh.get_hierarchy_of_triangular_meshes_for_sphere(
            splits=model_config.mesh_size
            )
finest_mesh = _meshes[-1]

mesh_phi, mesh_theta = model_utils.cartesian_to_spherical(
        finest_mesh.vertices[:, 0],
        finest_mesh.vertices[:, 1],
        finest_mesh.vertices[:, 2],
        )
mesh_nodes_lat, mesh_nodes_lon = model_utils.spherical_to_lat_lon(
        phi=mesh_phi, theta=mesh_theta
        )
num_mesh_nodes = finest_mesh.vertices.shape[0]  # Number of mesh nodes
# -----------------------------------------------------
# Init grid properties
# Sample data (replace with your actual data)
grid_lat = np.linspace(-90, 90, 300).astype(np.float32)  # Example grid latitude
grid_lon = np.linspace(-180, 180, 300).astype(np.float32)   # Example grid longitude
num_grid_nodes = grid_lat.shape[0] * grid_lon.shape[0] # Number of grid nodes
grid_nodes_lon, grid_nodes_lat = np.meshgrid(grid_lon, grid_lat)
grid_nodes_lat = grid_nodes_lat.reshape([-1]).astype(np.float32) # Example grid nodes latitude 
grid_nodes_lon = grid_nodes_lon.reshape([-1]).astype(np.float32)   # Example grid nodes longitude
# -----------------------------------------------------
spatial_features_kwargs = dict(
        add_node_positions=False,
        add_node_latitude=True,
        add_node_longitude=True,
        add_relative_positions=True,
        relative_longitude_local_coordinates=True,
        relative_latitude_local_coordinates=True,
    )  # Additional feature computation args (if any)
query_radius = 0.5  # Example query radius

# Create the graph
input_graph = init_grid2mesh_graph(
    grid_lat, grid_lon, finest_mesh, query_radius,
    grid_nodes_lat, grid_nodes_lon, mesh_nodes_lat, mesh_nodes_lon,
    num_grid_nodes, num_mesh_nodes, spatial_features_kwargs
)

In [68]:
input_graph

TypedGraph(context=Context(n_graph=array([1]), features=()), nodes={'grid_nodes': NodeSet(n_node=array([90000]), features=array([[-1.0000000e+00, -1.0000000e+00,  8.7422777e-08],
       [-1.0000000e+00, -9.9977922e-01, -2.1012342e-02],
       [-1.0000000e+00, -9.9911696e-01, -4.2015493e-02],
       ...,
       [ 1.0000000e+00, -9.9911696e-01,  4.2015493e-02],
       [ 1.0000000e+00, -9.9977922e-01,  2.1012342e-02],
       [ 1.0000000e+00, -1.0000000e+00, -8.7422777e-08]],
      shape=(90000, 3), dtype=float32)), 'mesh_nodes': NodeSet(n_node=array([642]), features=array([[ 0.18759258,  0.49999997,  0.86602545],
       [ 0.7946544 , -0.50000006,  0.8660254 ],
       [ 0.7946544 ,  1.        ,  0.        ],
       ...,
       [ 0.34392706, -0.7251567 ,  0.6885838 ],
       [ 0.24886788, -0.8127705 ,  0.58258396],
       [ 0.40267685, -0.8279131 ,  0.5608564 ]],
      shape=(642, 3), dtype=float32))}, edges={EdgeSetKey(name='grid2mesh', node_sets=('grid_nodes', 'mesh_nodes')): EdgeSet(n_ed

In [ ]:

_mesh = micosahedral_mesh.get_hierarchy_of_triangular_meshes_for_sphere(
            splits=model_config.mesh_size
            )
            )

4

In [ ]:
def _init_grid2mesh_graph(self) -> typed_graph.TypedGraph:
    """Build Grid2Mesh graph."""

    # Create some edges according to distance between mesh and grid nodes.
    assert self._grid_lat is not None and self._grid_lon is not None
    (grid_indices, mesh_indices) = grid_mesh_connectivity.radius_query_indices(
        grid_latitude=self._grid_lat,
        grid_longitude=self._grid_lon,
        mesh=self.._meshes[-1],
        radius=self._query_radius)

    # Edges sending info from grid to mesh.
    senders = grid_indices
    receivers = mesh_indices

    # Precompute structural node and edge features according to config options.
    # Structural features are those that depend on the fixed values of the
    # latitude and longitudes of the nodes.
    (senders_node_features, receivers_node_features,
     edge_features) = model_utils.get_bipartite_graph_spatial_features(
         senders_node_lat=self._grid_nodes_lat,
         senders_node_lon=self._grid_nodes_lon,
         receivers_node_lat=self._mesh_nodes_lat,
         receivers_node_lon=self._mesh_nodes_lon,
         senders=senders,
         receivers=receivers,
         edge_normalization_factor=None,
         **self._spatial_features_kwargs,
     )

    n_grid_node = np.array([self._num_grid_nodes])
    n_mesh_node = np.array([self._num_mesh_nodes])
    n_edge = np.array([mesh_indices.shape[0]])
    grid_node_set = typed_graph.NodeSet(
        n_node=n_grid_node, features=senders_node_features)
    mesh_node_set = typed_graph.NodeSet(
        n_node=n_mesh_node, features=receivers_node_features)
    edge_set = typed_graph.EdgeSet(
        n_edge=n_edge,
        indices=typed_graph.EdgesIndices(senders=senders, receivers=receivers),
        features=edge_features)
    nodes = {"grid_nodes": grid_node_set, "mesh_nodes": mesh_node_set}
    edges = {
        typed_graph.EdgeSetKey("grid2mesh", ("grid_nodes", "mesh_nodes")):
            edge_set
    }
    grid2mesh_graph = typed_graph.TypedGraph(
        context=typed_graph.Context(n_graph=np.array([1]), features=()),
        nodes=nodes,
        edges=edges)
    return grid2mesh_graph

In [ ]:
 def _run_grid2mesh_gnn(self, grid_node_features: chex.Array,
                         ) -> tuple[chex.Array, chex.Array]:
    """Runs the grid2mesh_gnn, extracting latent mesh and grid nodes."""

    # Concatenate node structural features with input features.
    batch_size = grid_node_features.shape[1]

    grid2mesh_graph = self._grid2mesh_graph_structure
    assert grid2mesh_graph is not None
    grid_nodes = grid2mesh_graph.nodes["grid_nodes"]
    mesh_nodes = grid2mesh_graph.nodes["mesh_nodes"]
    new_grid_nodes = grid_nodes._replace(
        features=jnp.concatenate([
            grid_node_features,
            _add_batch_second_axis(
                grid_nodes.features.astype(grid_node_features.dtype),
                batch_size)
        ],
                                 axis=-1))

    # To make sure capacity of the embedded is identical for the grid nodes and
    # the mesh nodes, we also append some dummy zero input features for the
    # mesh nodes.
    dummy_mesh_node_features = jnp.zeros(
        (self._num_mesh_nodes,) + grid_node_features.shape[1:],
        dtype=grid_node_features.dtype)
    new_mesh_nodes = mesh_nodes._replace(
        features=jnp.concatenate([
            dummy_mesh_node_features,
            _add_batch_second_axis(
                mesh_nodes.features.astype(dummy_mesh_node_features.dtype),
                batch_size)
        ],
                                 axis=-1))

    # Broadcast edge structural features to the required batch size.
    grid2mesh_edges_key = grid2mesh_graph.edge_key_by_name("grid2mesh")
    edges = grid2mesh_graph.edges[grid2mesh_edges_key]

    new_edges = edges._replace(
        features=_add_batch_second_axis(
            edges.features.astype(dummy_mesh_node_features.dtype), batch_size))

    input_graph = self._grid2mesh_graph_structure._replace(
        edges={grid2mesh_edges_key: new_edges},
        nodes={
            "grid_nodes": new_grid_nodes,
            "mesh_nodes": new_mesh_nodes
        })

    # Run the GNN.
    grid2mesh_out = self._grid2mesh_gnn(input_graph)
    latent_mesh_nodes = grid2mesh_out.nodes["mesh_nodes"].features
    latent_grid_nodes = grid2mesh_out.nodes["grid_nodes"].features
    return latent_mesh_nodes, latent_grid_nodes
